# 05 - Final Results Comparison & Report Visuals
### Customer Churn Prediction in the Banking Sector

This notebook consolidates everything from notebook 04 into **clean, publication-style
charts and tables** — ready to paste into the README, a portfolio writeup, or a slide deck.

It does **not** retrain anything — it only loads the saved metric CSVs and produces:

1. A single "headline" results table (best model, best accuracy, etc.)
2. Final Sample vs Cluster-average grouped bar charts (polished version of notebook 04's plots)
3. A model leaderboard sorted by F1-score
4. A short written conclusion (matching the paper's findings) generated from the actual numbers
5. All figures saved to `outputs/figures/` as PNGs for direct use in the README


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

FIG_DIR = "../outputs/figures"
os.makedirs(FIG_DIR, exist_ok=True)


## 1. Load Saved Metrics (from notebook 04)

In [ ]:
results_original = pd.read_csv("../outputs/metrics/results_original.csv")
results_clusters = pd.read_csv("../outputs/metrics/results_clusters.csv")
comparison = pd.read_csv("../outputs/metrics/sample_vs_cluster_avg.csv")

model_order = ['KNN', 'LR', 'DT', 'RF', 'SVM']
metrics = ['Accuracy', 'Precision', 'Recall', 'F1_Score']

results_original = results_original.set_index('Model').reindex(model_order).reset_index()
comparison = comparison.set_index('Model').reindex(model_order).reset_index()

print("Results (Original / no segmentation):")
results_original


## 2. Headline Results Table

In [ ]:
best_overall = results_original.sort_values('F1_Score', ascending=False).iloc[0]

print("=" * 55)
print(" HEADLINE RESULT")
print("=" * 55)
print(f" Best model:      {best_overall['Model']}")
print(f" Accuracy:        {best_overall['Accuracy']}%")
print(f" Precision:       {best_overall['Precision']}%")
print(f" Recall:          {best_overall['Recall']}%")
print(f" F1-score:        {best_overall['F1_Score']}%")
print("=" * 55)


In [ ]:
# Full leaderboard, sorted by F1
leaderboard = results_original.sort_values('F1_Score', ascending=False).reset_index(drop=True)
leaderboard.index = leaderboard.index + 1
leaderboard.index.name = 'Rank'
leaderboard


## 3. Model Leaderboard Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(leaderboard))
width = 0.2
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for i, metric in enumerate(metrics):
    ax.bar(x + i * width, leaderboard[metric], width, label=metric.replace('_', ' '), color=colors[i])

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(leaderboard['Model'])
ax.set_ylim(0, 105)
ax.set_ylabel('Score (%)')
ax.set_title('Model Performance Leaderboard (Full Dataset, No Segmentation)')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/model_leaderboard.png")
plt.show()


## 4. Sample vs Cluster Average — Final Comparison Charts

Reproduces the paper's Figures 5-8, polished for the README.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, metric in zip(axes, metrics):
    x = np.arange(len(comparison))
    width = 0.35

    bars1 = ax.bar(x - width/2, comparison[f'{metric}_Sample'], width,
                    label='Sample (no segmentation)', color='#4C72B0')
    bars2 = ax.bar(x + width/2, comparison[f'{metric}_ClusterAvg'], width,
                    label='Cluster average (with segmentation)', color='#DD8452')

    for bars in (bars1, bars2):
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(comparison['Model'])
    ax.set_title(f'{metric.replace("_", " ")}: Sample vs Cluster Average')
    ax.set_ylabel(f'{metric.replace("_", " ")} (%)')
    ax.set_ylim(0, 108)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/sample_vs_cluster_avg.png")
plt.show()


## 5. Per-Cluster Breakdown Heatmap (Accuracy)

In [ ]:
pivot_acc = results_clusters.pivot(index='Model', columns='Cluster', values='Accuracy')
pivot_acc = pivot_acc.reindex(model_order)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_acc, annot=True, fmt='.2f', cmap='YlGnBu', cbar_kws={'label': 'Accuracy (%)'})
plt.title('Accuracy (%) by Model and Cluster')
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/accuracy_heatmap_by_cluster.png")
plt.show()


## 6. Segmentation Impact Summary

For each metric, what fraction of models performed *worse* after segmentation
(cluster average < sample)? This reproduces the paper's "3:5, 2:5, 4:5, 3:5" finding.


In [ ]:
impact_summary = {}
for metric in metrics:
    worse_after_segmentation = (comparison[f'{metric}_ClusterAvg'] < comparison[f'{metric}_Sample']).sum()
    total = len(comparison)
    impact_summary[metric] = f"{worse_after_segmentation}:{total}"

impact_df = pd.DataFrame.from_dict(impact_summary, orient='index', columns=['Models worse after segmentation'])
impact_df


## 7. Auto-Generated Written Conclusion

In [ ]:
best_model_name = best_overall['Model']
best_acc = best_overall['Accuracy']
best_f1 = best_overall['F1_Score']

avg_sample_acc = comparison['Accuracy_Sample'].mean()
avg_cluster_acc = comparison['Accuracy_ClusterAvg'].mean()
diff = avg_cluster_acc - avg_sample_acc
direction = "improved" if diff > 0 else "did not improve" if diff < 0 else "had no effect on"

conclusion = f"""
**Conclusion:**

Across all 5 models tested, **{best_model_name}** achieved the best overall performance on the
full (non-segmented) dataset, with **{best_acc}% accuracy** and an **F1-score of {best_f1}%**.

When comparing the average performance across the 6 customer segments ("Cluster average")
against the full dataset ("Sample"), customer segmentation **{direction}** average accuracy
(Sample: {avg_sample_acc:.2f}% vs Cluster average: {avg_cluster_acc:.2f}%, a difference of
{diff:+.2f} percentage points).

This supports the original paper's finding: **customer segmentation does not consistently
improve churn prediction accuracy** — its effect depends heavily on the dataset and the
specific model used, rather than being a guaranteed improvement.
"""

print(conclusion)

with open("../outputs/metrics/conclusion.md", "w") as f:
    f.write(conclusion)


## 8. Export Final Summary Table (for README)

In [ ]:
readme_table = leaderboard[['Model', 'Accuracy', 'Precision', 'Recall', 'F1_Score']]
markdown_table = readme_table.to_markdown(index=False)
print(markdown_table)

with open("../outputs/metrics/results_table.md", "w") as f:
    f.write(markdown_table)

print("\nSaved markdown table to outputs/metrics/results_table.md")
print("Saved figures to outputs/figures/")


## Summary

This notebook produced the final, polished deliverables for the project:

- `outputs/figures/model_leaderboard.png`
- `outputs/figures/sample_vs_cluster_avg.png`
- `outputs/figures/accuracy_heatmap_by_cluster.png`
- `outputs/metrics/results_table.md` (paste directly into README)
- `outputs/metrics/conclusion.md` (auto-generated written conclusion)

**Project pipeline recap:**
`Raw data → EDA → Preprocessing → K-Means Segmentation → SMOTE + 5 Models → Final Comparison`

➡️ **Optional next steps:**
- Refactor the notebook logic into reusable modules under `src/` (`data_loader.py`,
  `preprocessing.py`, `clustering.py`, `models.py`, `evaluate.py`)
- Build `app/streamlit_app.py` — an interactive churn predictor using the saved
  `models/saved_models/best_model_*.pkl`
- Write the final `README.md` using the saved table/figures above
